# scinterop — testing on real PBMC data

A walkthrough that downloads PBMC3K, reads it into the canonical format,
converts between H5AD / MTX / RDS, and checks everything along the way.

In [1]:
import scinterop as si
import scanpy as sc
import numpy as np
import pandas as pd
from pathlib import Path

print("scinterop", si.__version__)
print("scanpy", sc.__version__)
print("numpy", np.__version__)
print("pandas", pd.__version__)

scinterop 0.1.0
scanpy 1.12.2
numpy 2.4.6
pandas 3.0.3


/tmp/ipykernel_3414870/2508672616.py:8: FutureWarning: `__version__` is deprecated, use `importlib.metadata.version('scanpy')` instead
  print("scanpy", sc.__version__)


In [2]:
# paths — everything lives inside the project folder
BASE = Path.cwd().parent  # tutorial/ is inside scinterop/
DATA_DIR = BASE / "data"
OUTPUT_DIR = BASE / "OUTPUT"
DATA_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Data:", DATA_DIR)
print("Out: ", OUTPUT_DIR)

Data: /exafs1/well/rittscher/users/qwi813/scinterop/data
Out:  /exafs1/well/rittscher/users/qwi813/scinterop/OUTPUT


---
## 1. Get some real data

Download the PBMC3K dataset that ships with scanpy and cache it in `data/`.

In [3]:
adata = sc.datasets.pbmc3k()
adata.write(DATA_DIR / "pbmc3k.h5ad")
print(f"AnnData: {adata.n_obs} cells x {adata.n_vars} genes")

AnnData: 2700 cells x 32738 genes


---
## 2. What format is this?

In [4]:
d = si.detect(DATA_DIR / "pbmc3k.h5ad")
print(f"Format  : {d.fmt}")
print(f"Details : {d.details}")

Format  : h5ad
Details : {'extension': '.h5ad'}


---
## 3. Read into CanonicalObject

In [5]:
obj = si.read(DATA_DIR / "pbmc3k.h5ad")
print(f"Shape   : {obj.shape}")
print(f"Obs cols: {list(obj.obs.columns)}")
print(f"Var cols: {list(obj.var.columns)}")
print(f"Obsp    : {list(obj.obsm.keys())}")
print(f"Layers  : {list(obj.layers.keys())}")
print(f"Uns keys: {list(obj.uns.keys())[:5]}")

Shape   : (2700, 32738)
Obs cols: []
Var cols: ['gene_ids']
Obsp    : []
Layers  : [None]
Uns keys: []


In [6]:
obs_head = obj.obs.head()
var_head = obj.var.head()
display(obs_head)
print(f"... ({len(obj.obs)} rows total)")
display(var_head)
print(f"... ({len(obj.var)} rows total)")

""
index
AAACATACAACCAC-1
AAACATTGAGCTAC-1
AAACATTGATCAGC-1
AAACCGTGCTTCCG-1
AAACCGTGTATGCG-1


... (2700 rows total)


,gene_ids
index,
MIR1302-10,ENSG00000243485
FAM138A,ENSG00000237613
OR4F5,ENSG00000186092
RP11-34P13.7,ENSG00000238009
RP11-34P13.8,ENSG00000239945


... (32738 rows total)


---
## 4. Validate

In [7]:
result = si.validate(obj)
if result.valid:
    print("Everything looks good.")
else:
    print("Issues found:", result.errors)

Everything looks good.


---
## 5. Convert to MTX (10X format)

This writes three files: `matrix.mtx`, `barcodes.tsv`, `features.tsv`.

In [8]:
mtx_out = OUTPUT_DIR / "pbmc3k_mtx"
si.convert(DATA_DIR / "pbmc3k.h5ad", mtx_out)
for f in sorted(mtx_out.iterdir()):
    print(f.name)

barcodes.tsv
features.tsv
matrix.mtx


Read it back and check the data survived.

In [9]:
obj2 = si.read(mtx_out)
print(f"Read back shape: {obj2.shape}")

match = np.allclose(obj.X[:5, :5].toarray(), obj2.X[:5, :5].toarray())
print(f"First 5x5 values match: {match}")

Read back shape: (2700, 32738)
First 5x5 values match: True


---
## 6. Round-trip back to H5AD

In [10]:
h5ad_out = OUTPUT_DIR / "pbmc3k_roundtrip.h5ad"
si.convert(mtx_out, h5ad_out)
obj3 = si.read(h5ad_out)
print(f"Round-trip shape: {obj3.shape}")
match2 = np.allclose(obj.X.toarray(), obj3.X.toarray())
print(f"Full matrix matches: {match2}")

Round-trip shape: (2700, 32738)


Full matrix matches: True


---
## 7. Convert to Seurat (RDS)

This step needs Rscript and the `anndata` R package.
If they aren't available the cell prints setup instructions instead of crashing.

In [11]:
import shutil
rds_out = OUTPUT_DIR / "pbmc3k.rds"

if shutil.which("Rscript"):
    try:
        si.convert(DATA_DIR / "pbmc3k.h5ad", rds_out, seurat=True)
        print(f"Seurat object saved — {rds_out.stat().st_size / 1e6:.1f} MB")
    except Exception as e:
        print(f"R bridge failed:\n{e}")
        print("\nTry installing the R anndata package:")
        print("  R -e 'install.packages(\"anndata\", repos=\"https://cloud.r-project.org\")'")
else:
    print("Rscript not found — skip RDS for now.")
    print("To enable:")
    print("  1. Install R")
    print("  2. R -e 'install.packages(\"anndata\", repos=\"https://cloud.r-project.org\")'")

Seurat object saved — 5.1 MB


---
## 8. Detect every output

Run `detect()` on everything we produced.

In [12]:
for p in sorted(OUTPUT_DIR.rglob("*")):
    if p.is_file() or p.is_dir():
        try:
            d = si.detect(p)
            print(f"{d.fmt:6s}  {p.relative_to(BASE)}")
        except si.errors.DetectionError:
            pass

rds     OUTPUT/final_list.rds
rds     OUTPUT/final_seurat.rds
h5ad    OUTPUT/obj_method_test.h5ad
mtx     OUTPUT/obj_method_test_mtx
mtx     OUTPUT/obj_method_test_mtx/matrix.mtx
rds     OUTPUT/pbmc3k.rds
rds     OUTPUT/pbmc3k_list.rds
mtx     OUTPUT/pbmc3k_mtx
mtx     OUTPUT/pbmc3k_mtx/matrix.mtx
h5ad    OUTPUT/pbmc3k_roundtrip.h5ad
rds     OUTPUT/pbmc3k_seurat.rds
h5ad    OUTPUT/pbmc3k_seurat_roundtrip.h5ad
rds     OUTPUT/sparse_list.rds
rds     OUTPUT/sparse_seurat.rds


---
## 9. Provenance records

Every conversion automatically wrote a JSON line to a provenance log.  Find and show it.

In [13]:
import json
scratch = Path.home() / "tmp" / f"scinterop_{__import__('os').environ.get('USER', 'unknown')}"
if not scratch.exists():
    scratch = Path.home() / "tmp"
provenance_files = sorted(Path("/tmp").rglob("provenance.jsonl")) + sorted(Path.home().rglob("provenance.jsonl"))

if provenance_files:
    log = provenance_files[0]
    print(f"Found: {log}")
    with open(log) as f:
        for line in f:
            r = json.loads(line)
            print(f"  {r['input_format']} → {r['output_format']}  "
                  f"ok={r['success']}  {r['runtime_seconds']:.1f}s  "
                  f"{r['timestamp'][:19]}")
else:
    print("No provenance logs found yet — they'll appear after a conversion runs.")

Found: /tmp/tmp0o25lae5/provenance.jsonl
  h5ad → rds  ok=True  1.0s  2026-07-10T09:02:29
  h5ad → rds  ok=False  0.5s  2026-07-10T09:02:29


---
## 10. Object convenience methods

Same conversions, but called directly on the CanonicalObject.

In [14]:
obj.to_mtx(OUTPUT_DIR / "obj_method_test_mtx")
obj.to_anndata(OUTPUT_DIR / "obj_method_test.h5ad")
print("MTX and H5AD written via .to_mtx() / .to_anndata()")

MTX and H5AD written via .to_mtx() / .to_anndata()


---
## 11. Final sanity — run the test suite

In [15]:
import sys, subprocess
result = subprocess.run(
    [sys.executable, "-m", "pytest", str(BASE / "scinterop" / "tests"), "-v", "--tb=short"],
    capture_output=True, text=True, timeout=120,
)
print(result.stdout[-1500:] if len(result.stdout) > 1500 else result.stdout)
if result.returncode == 0:
    print("All tests passed.")
else:
    print(result.stderr[-500:])

t_layers_shape_mismatch PASSED [ 91%]
../scinterop/tests/test_validate.py::TestValidate::test_assert_valid_passes PASSED [ 93%]
../scinterop/tests/test_validate.py::TestValidate::test_assert_valid_raises PASSED [ 95%]
../scinterop/tests/test_validate.py::TestValidate::test_require_raw PASSED [ 97%]
../scinterop/tests/test_validate.py::TestValidate::test_valid_with_raw PASSED [100%]

=============================== warnings summary ===============================
scinterop/tests/test_mtx.py::TestMtxIO::test_write_and_read_roundtrip
scinterop/tests/test_mtx.py::TestMtxIO::test_read_single_mtx_file
  /exafs1/well/rittscher/users/qwi813/scinterop/scinterop/mtx.py:150: DeprecationWarning: The default value for `spmatrix` is changing to `False` in v1.20.
               That means the default return type will be a sparse array.
               Unless you use * instead of @, ** for matrix power, or you depend
               on 2D shapes from e.g. `A.sum(axis=0)` it may not matter to you.
      

---
## Done

Check the `OUTPUT/` folder — you'll find:
- `pbmc3k_mtx/` — 10X-format MTX directory
- `pbmc3k_roundtrip.h5ad` — H5AD after an MTX round-trip
- `pbmc3k.rds` — Seurat object (if R was available)
- `obj_method_test_mtx/` and `obj_method_test.h5ad` — written via convenience methods

The `data/` folder keeps the original PBMC3K H5AD so you don't re-download next time.